# Introduction

Le football est un domaine emblématique de la recherche en intelligence artificielle, car il combine prise de décision en temps réel, stratégie collective et environnement dynamique. Pour permettre l’expérimentation et l’entraînement d’agents intelligents dans ce contexte, Google Research a développé **Google Research Football (GFootball)**, un environnement de simulation open source qui reproduit les principaux aspects d’un match de football. 

GFootball s’impose aujourd’hui comme un benchmark de référence pour le **reinforcement learning (apprentissage par renforcement)** : il offre un environnement complexe, riche en interactions, où les agents doivent apprendre à coopérer, à anticiper et à s’adapter face à des adversaires, tout en maximisant leur performance (nombre de buts, passes réussies, etc.).

---

**Dans ce projet**, nous explorons les fondamentaux du reinforcement learning à travers GFootball.  
Nous commencerons par un agent simple à base de règles ("rule-based agent"), puis nous entraînerons un agent d’apprentissage profond (deep RL) sur l’environnement, afin de comparer leurs performances et de mieux comprendre les principes de l’apprentissage par renforcement appliqués à un problème concret et réaliste.

L’objectif est à la fois de mettre en œuvre des techniques modernes de RL, et de développer une compréhension fine du fonctionnement, des forces et des limites de ces approches dans un environnement simulé de haut niveau.

# 🏁 1- Prise en main de GFootball : 


In [13]:
import os 
import gfootball.env as football_env
import gym # install 0.25.2 because > is not compatible with gfootball
import numpy as np
import matplotlib.pyplot as plt 
import time


os.makedirs("dumps", exist_ok=True)
os.chdir("dumps")

In [14]:
# Create the environment 
env = football_env.create_environment(
    env_name='11_vs_11_stochastic',  # Match complet
    representation='simple115',      # Observation sous forme de vecteurs (plus facile à manipuler)
    render=False                     # Mettre True pour visualiser (à faire en local)
)

In [15]:
obs = env.reset()
print("Observation shape:", np.array(obs).shape)
print("Sample observation:", obs)

print("Action space:", env.action_space)
print("Number of possible actions:", env.action_space.n)

Observation shape: (115,)
Sample observation: [-1.0110294  -0.          0.          0.02032536  0.         -0.02032536
 -0.4266544  -0.19894461 -0.5055147  -0.06459399 -0.5055147   0.06459298
 -0.4266544   0.19894461 -0.18624374 -0.10739919 -0.2705252  -0.
 -0.18624374  0.10739919 -0.01011029 -0.2196155   0.         -0.
  0.         -0.          0.         -0.          0.         -0.
  0.         -0.          0.         -0.          0.         -0.
  0.         -0.          0.         -0.          0.         -0.
  0.         -0.          1.0110294   0.          0.05055147  0.
  0.01011029 -0.21961753  0.4266544   0.19894461  0.5055147   0.06459399
  0.5055147  -0.06459298  0.4266544  -0.19894461  0.18624374  0.10739919
  0.2705252   0.          0.18624374 -0.10739919  0.01011029  0.2196155
 -0.          0.         -0.          0.         -0.          0.
 -0.          0.         -0.          0.         -0.          0.
 -0.          0.         -0.          0.         -0.          0.
 -0. 

## ⚽️ GFootball – Observations & Actions Reference

### **Raw Observations**

The GFootball environment provides a comprehensive observation vector representing the current state of the match.  
Below is a breakdown of the main elements available in the **raw observation**:

- **Ball information:**
  - `ball`: [x, y, z] position of the ball
  - `ball_direction`: [x, y, z] movement vector of the ball
  - `ball_rotation`: [x, y, z] rotation angles (radians)
  - `ball_owned_team`: {-1, 0, 1}, (-1: not owned, 0: left team, 1: right team)
  - `ball_owned_player`: Index of the player (0..N-1) owning the ball

- **Left team:**
  - `left_team`: (N x 2) [x, y] positions of each player
  - `left_team_direction`: (N x 2) [x, y] velocity vectors of players
  - `left_team_tired_factor`: (N) Fatigue level, 0 = fresh, 1 = fully tired
  - `left_team_yellow_card`: (N) 0/1 if player has a yellow card
  - `left_team_active`: (N) Boolean, True if player is on the field
  - `left_team_roles`: (N) Player roles, e.g., GK, CB, LB, etc.

- **Right team:** (Same structure as left team)
  - `right_team`, `right_team_direction`, `right_team_tired_factor`, `right_team_yellow_card`, `right_team_active`, `right_team_roles`

- **Controlled player information:**
  - `active`: Index of currently controlled player
  - `designated`: Index of the main player (usually equals active)
  - `sticky_actions`: 10-element vector indicating which movement/action keys are being held

- **Match state:**
  - `score`: [left_team_goals, right_team_goals]
  - `steps_left`: Steps remaining in the match
  - `game_mode`: Game situation (KickOff, GoalKick, FreeKick, etc.)

- **Screen:**
  - `frame`: (Optional) RGB pixel image of the game (only if rendering is enabled)

> **Note:**  
> Field coordinates:  
> - Bottom left/right corners: `[-1, 0.42]` and `[1, 0.42]`  
> - Top left/right: `[-1, -0.42]` and `[1, -0.42]`  
> - Goals: X = -1 (left), X = 1 (right), Y in [-0.044, 0.044]

---

### **Observation Wrappers**

The environment provides convenient wrappers to simplify observations:

- **simple115 / simple115v2:**  
  115 floats encoding positions, directions, ball info, active player, game mode, etc.  
  `simple115v2` fixes a bug present in `simple115` for better robustness.

- **extracted (SMMWrapper):**  
  Minimap-like 2D "planes" (images) showing positions of players and ball.

- **pixels / pixels_gray:**  
  Downscaled raw pixel image or grayscale version.

- **SingleAgentWrapper:**  
  Removes first dimension if only one agent is being controlled (useful for single-agent RL).

---

### **Action Space: Default Action Set**

At each time step, the agent must choose one action from the following set (19 possible actions):

| Action Index | Name                | Description                                                       |
|--------------|---------------------|-------------------------------------------------------------------|
| 0            | `idle`              | No action; maintain current movement                              |
| 1-8          | `left` ... `bottom_left` | Move in 8 directions (sticky actions)                        |
| 9            | `long_pass`         | Long ground pass to a teammate                                    |
| 10           | `high_pass`         | High pass to a teammate                                           |
| 11           | `short_pass`        | Short ground pass                                                 |
| 12           | `shot`              | Shoot towards opponent’s goal                                     |
| 13           | `sprint`            | Sprint (sticky)                                                   |
| 14           | `release_direction` | Stop current movement direction                                   |
| 15           | `release_sprint`    | Stop sprinting                                                    |
| 16           | `sliding`           | Slide tackle (when not in possession)                             |
| 17           | `dribble`           | Start dribbling (sticky, harder to dispossess)                    |
| 18           | `release_dribble`   | Stop dribbling                                                    |

> For extended actions or detailed info, see the [GFootball documentation](https://github.com/google-research/football/blob/master/docs/actions.md).

---

### **Recommended Wrappers**

- For most RL experiments, **use `simple115v2` or `simple115`** as the observation type for a concise, consistent state representation.
- **Discrete(19)** is the standard action space, matching the table above.

---

### **References**

- [GFootball Observation Documentation](https://github.com/google-research/football/blob/master/gfootball/doc/observation.md)

---


## Random Agent in GFootball with 2D Visualization


In [16]:
# 1. Create the GFootball environment with 2D visualization enabled.
env = football_env.create_environment(
    env_name='11_vs_11_stochastic',     # Full 11v11 match with stochasticity
    representation='simple115v2',       # Improved state vector (robust version)
    render=False,                         # Enables 2D visualization (window will open)
    logdir="dumps",
    write_full_episode_dumps= True, # Disable saving 3D video files
    write_video = True # Disable writing 3D mp4 video
)


In [17]:
# 2. Reset the environment to start a new match.
obs = env.reset()

done = False        # This flag becomes True when the match ends
total_reward = 0    # We accumulate the rewards to see how the random agent performs
step_count = 0      # Number of steps in the match

In [18]:
# 3. Run a single episode (one match) with a random agent.
print("Starting a random agent match... (Close the 2D window to finish)")

while not done:
    # The agent samples a random action from the available action space (Discrete(19))
    action = env.action_space.sample()
    
    # Take a step in the environment using the chosen action
    obs, reward, done, info = env.step([action])
    
    # Accumulate the total reward
    total_reward += reward
    step_count += 1
    
    # Optional: Add a tiny delay so the 2D window doesn't move too fast
    time.sleep(0.03)
    
    # Print info every 100 steps
    if step_count % 100 == 0:
        print(f"Step: {step_count} | Current reward: {total_reward}")

# 4. Close the environment to clean up resources.
env.close()

# 5. Show final stats
print("\nMatch ended!")
print(f"Total steps: {step_count}")
print(f"Total reward: {total_reward}")

Starting a random agent match... (Close the 2D window to finish)
Step: 100 | Current reward: 0.0
Step: 200 | Current reward: 0.0
Step: 300 | Current reward: 0.0
Step: 400 | Current reward: -1.0
Step: 500 | Current reward: -1.0
Step: 600 | Current reward: -1.0
Step: 700 | Current reward: -1.0
Step: 800 | Current reward: -1.0
Step: 900 | Current reward: -1.0
Step: 1000 | Current reward: -1.0
Step: 1100 | Current reward: -1.0
Step: 1200 | Current reward: -1.0
Step: 1300 | Current reward: -1.0
Step: 1400 | Current reward: -1.0
Step: 1500 | Current reward: -1.0
Step: 1600 | Current reward: -1.0
Step: 1700 | Current reward: -1.0
Step: 1800 | Current reward: -1.0
Step: 1900 | Current reward: -1.0
Step: 2000 | Current reward: -1.0
Step: 2100 | Current reward: -2.0
Step: 2200 | Current reward: -2.0
Step: 2300 | Current reward: -2.0
Step: 2400 | Current reward: -2.0
Step: 2500 | Current reward: -3.0
Step: 2600 | Current reward: -3.0
Step: 2700 | Current reward: -3.0
Step: 2800 | Current reward: 

### ==> Now let's change the reward system for example : 

In [19]:
import plotly.graph_objects as go

fig = go.Figure()

# Tracer le terrain
fig.add_shape(type="rect", x0=-1, x1=1, y0=-0.42, y1=0.42,
              line=dict(color="black", width=2))

# Zones colorées
zones = [
    {"x0": -1.0, "x1": -0.5,"y0":-0.42,"y1":0.42, "color": "lightblue", "label": "Défense"},
    {"x0": -0.5, "x1": 0,"y0":-0.42,"y1":0.42, "color": "lightgreen", "label": "Milieu_def"},
    {"x0": 0.5, "x1": 0,"y0":-0.42,"y1":0.42, "color": "green", "label": "Milieu_att"},
    {"x0": 0.5, "x1": 1.0,"y0":-0.2,"y1":0.2, "color": "salmon", "label": "Attaque_centre"},
    {"x0": 0.5, "x1": 1.0,"y0":0.2,"y1":0.42, "color": "grey", "label": "Attaque_ailes_g"},
    {"x0": 0.5, "x1": 1.0,"y0":-0.2,"y1":-0.42, "color": "yellow", "label": "Attaque_ailes_d"},
]

for zone in zones:
    fig.add_shape(type="rect", x0=zone["x0"], x1=zone["x1"], y0=zone["y0"], y1=zone["y1"],
                  fillcolor=zone["color"], opacity=0.3, line_width=0)

# Ligne centrale
fig.add_shape(type="line", x0=0, x1=0, y0=-0.42, y1=0.42,
              line=dict(color="gray", dash="dash"))

# Mise en page
fig.update_layout(title="Terrain GFootball (zones interactives)",
                  xaxis=dict(range=[-1.1, 1.1], zeroline=False),
                  yaxis=dict(range=[-0.5, 0.5], zeroline=False),
                  width=900, height=450,
                  showlegend=False,
                  template="plotly_white")

fig.show()

In [20]:
from enum import Enum

# Enum pour représenter les équipes
class BallOwnership(Enum):
    NOONE = -1
    LEFT = 0
    RIGHT = 1

# Enum pour les modes de jeu
class GameMode(Enum):
    NORMAL = 0
    KICKOFF = 1
    GOAL_KICK = 2
    FREE_KICK = 3
    CORNER = 4
    THROW_IN = 5
    PENALTY = 6

# Enum pour les indices précis de l'observation simple115v2
class Simple115v2Indices(Enum):
    LEFT_TEAM_POS = slice(0, 22)
    LEFT_TEAM_DIR = slice(22, 44)
    RIGHT_TEAM_POS = slice(44, 66)
    RIGHT_TEAM_DIR = slice(66, 88)
    BALL_POSITION = slice(88, 91)
    BALL_DIRECTION = slice(91, 94)
    BALL_OWNERSHIP = slice(94, 97)
    ACTIVE_PLAYER = slice(97, 108)
    GAME_MODE = slice(108, 115)


In [9]:
import gym 
import numpy as np 
import logging 
import time
from datetime import datetime

# === Setup Logging ===
log_filename = f"random_agent_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
logging.basicConfig(
    filename=log_filename,
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt='%H:%M:%S'
)

# Also print to console
console = logging.StreamHandler()
console.setLevel(logging.INFO)
formatter = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", datefmt='%H:%M:%S')
console.setFormatter(formatter)
logging.getLogger('').addHandler(console)

class MyRewardWrapper(gym.Wrapper) : 
    def __init__(self, env):
        super(MyRewardWrapper,self).__init__(env)
        self.prev_obs = None
        self.prev_score = [0, 0]

    def reset(self) : 
        self.prev_obs = self.env.reset()
        self.prev_score = [0, 0]  # reset score tracking
        return self.prev_obs

    def step(self,action) :
        obs, reward, done, info = self.env.step(action)
        custom_reward = reward
        raw_obs = info.get("raw_obs", {})
        zone_reward = self._custom_zone_reward(self.prev_obs, obs, action, raw_obs)
        goal_reward = self._goal_scored(info)
        custom_reward += zone_reward + goal_reward

        logging.info(f"Step | Action: {action} | Zone reward: {zone_reward:.2f} | Goal reward: {goal_reward:.2f} | Total reward: {custom_reward:.2f}")

        self.prev_obs = obs 
        return obs, custom_reward, done, info  

    def _custom_zone_reward(self, prev_obs, obs, action, raw_obs):
        reward = 0.0
        ball_x, ball_y = obs[Simple115v2Indices.BALL_POSITION.value][:2]
        zone = self._get_zone(ball_x, ball_y)
        ball_owner_team = np.argmax(obs[Simple115v2Indices.BALL_OWNERSHIP.value]) - 1
        active_player = np.argmax(obs[Simple115v2Indices.ACTIVE_PLAYER.value])

        if zone is None:
            return 0.0

        if ball_owner_team != 0:
            return 0.0

        if zone == "defense":
            if self._successful_pass(prev_obs, obs):
                reward += 1
                logging.info("✅ Short pass in DEFENSE zone (+0.1)")
            if self._ball_recovered(prev_obs, obs):
                reward += 10
                logging.info("✅ Ball recovered in DEFENSE zone (+10)")
            if self._lost_possession(prev_obs, obs):
                reward -= 10.0
                logging.info("❌ Lost possession in DEFENSE zone (-5.0)")
            if self._idle_or_back_pass(prev_obs, obs):
                reward -= 0.1
                logging.info("⚠️ Passive behavior in DEFENSE zone (-1.0)")

        elif zone == "milieu_def":
            if self._short_forward_pass(prev_obs, obs):
                reward += 10
                logging.info("✅ Forward pass in MILIEU_DEF zone (+10)")
            if self._successful_pass(prev_obs, obs):
                reward += 5
                logging.info("✅ Short pass in MILIEU_DEF zone (+5)")
            if self._ball_recovered(prev_obs, obs):
                reward += 10
                logging.info("✅ Ball recovered in MILIEU_DEF zone (+20)")
            if self._lost_possession(prev_obs, obs):
                reward -= 5
                logging.info("❌ Lost possession in MILIEU_DEF zone (-5)")

        elif zone == "milieu_att":
            if self._short_forward_pass(prev_obs, obs):
                reward += 10
                logging.info("✅ Forward pass in MILIEU_ATT zone (+20)")
            if self._successful_pass(prev_obs, obs):
                reward += 5
                logging.info("✅ Ball recovered in MILIEU_ATT zone (+1)")
            if self._lost_possession(prev_obs, obs):
                reward -= 1
                logging.info("❌ Lost possession in MILIEU_ATT zone (-1)")

        elif zone in ["attaque_ailes_g", "attaque_ailes_d"]:
            if self._dribble(raw_obs):
                reward += 30
                logging.info("✅ Dribble on the wing (+30)")
            if self._successful_pass(prev_obs, obs):
                reward += 2
                logging.info("✅ Ball recovered on the wing (+1)")
            if self._lost_possession(prev_obs, obs):
                reward -= 1
                logging.info("❌ Lost possession on the wing (-1)")

        elif zone == "attaque_centre":
            if self._shot_taken(action):
                reward += 50
                logging.info("✅ Shot taken in ATTACK CENTRE zone (+50)")
            if self._successful_pass(prev_obs, obs):
                reward += 5
                logging.info("✅ Ball recovered in ATTACK CENTRE zone (+0.1)")
            if self._dribble(raw_obs):
                reward += 5
                logging.info("❌ Lost possession in ATTACK CENTRE zone (-0.2)")

        return reward

    

# ================================= HELPERS ====================================================

    def _successful_pass(self, prev_obs, obs):
        prev_ball_owner_team = np.argmax(prev_obs[Simple115v2Indices.BALL_OWNERSHIP.value]) - 1
        curr_ball_owner_team = np.argmax(obs[Simple115v2Indices.BALL_OWNERSHIP.value]) - 1
        prev_ball_owner_player = np.argmax(prev_obs[Simple115v2Indices.ACTIVE_PLAYER.value])
        curr_ball_owner_player = np.argmax(obs[Simple115v2Indices.ACTIVE_PLAYER.value])
        return (prev_ball_owner_team == curr_ball_owner_team and
                prev_ball_owner_team != -1 and
                prev_ball_owner_player != curr_ball_owner_player)

    def _get_zone(self, x, y):
        if -1.0 <= x < -0.5:
            return "defense"
        elif -0.5 <= x < 0:
            return "milieu_def"
        elif 0 <= x < 0.5:
            return "milieu_att"
        elif 0.5 <= x <= 1.0:
            if -0.2 <= y <= 0.2:
                return "attaque_centre"
            elif y > 0.2:
                return "attaque_ailes_g"
            elif y < -0.2:
                return "attaque_ailes_d"
        return None
    
    def _goal_scored(self, info):
        score = info.get("score", [0, 0])
        if score[0] > self.prev_score[0]:
            logging.info("⚽ Goal scored! (+100 reward)")
            self.prev_score = score
            return 100.0
        self.prev_score = score
        return 0.0
    def _lost_possession(self, prev_obs, obs):
        prev_team = np.argmax(prev_obs[Simple115v2Indices.BALL_OWNERSHIP.value]) - 1
        curr_team = np.argmax(obs[Simple115v2Indices.BALL_OWNERSHIP.value]) - 1
        return prev_team == 0 and curr_team != 0

    def _ball_recovered(self, prev_obs, obs):
        prev_team = np.argmax(prev_obs[Simple115v2Indices.BALL_OWNERSHIP.value]) - 1
        curr_team = np.argmax(obs[Simple115v2Indices.BALL_OWNERSHIP.value]) - 1
        return prev_team != 0 and curr_team == 0

    def _short_pass(self, prev_obs, obs):
        prev_owner = np.argmax(prev_obs[Simple115v2Indices.ACTIVE_PLAYER.value])
        curr_owner = np.argmax(obs[Simple115v2Indices.ACTIVE_PLAYER.value])
        return prev_owner != curr_owner

    def _short_forward_pass(self, prev_obs, obs):
        prev_ball_x = prev_obs[Simple115v2Indices.BALL_POSITION.value][0]
        curr_ball_x = obs[Simple115v2Indices.BALL_POSITION.value][0]
        return curr_ball_x - prev_ball_x > 0.05

    def _idle_or_back_pass(self, prev_obs, obs):
        prev_x = prev_obs[Simple115v2Indices.BALL_POSITION.value][0]
        curr_x = obs[Simple115v2Indices.BALL_POSITION.value][0]
        return curr_x - prev_x < 0

    def _dribble(self, raw_obs):
        return raw_obs.get("sticky_actions", [0]*10)[9] == 1

    def _shot_taken(self, action):
        return action == 12


In [ ]:
import time
import gym
import logging
from datetime import datetime
from gfootball.env import create_environment

# === Logging Setup ===
now = datetime.now().strftime("%Y%m%d_%H%M%S")
log_filename = f"match_logs_{now}.log"

logging.basicConfig(
    filename=log_filename,
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# === Create the Football Environment ===
env = create_environment(
    env_name="11_vs_11_stochastic",
    representation="simple115v2",
    render=False,
    logdir="dumps",
    write_full_episode_dumps=True,
    write_video=True
)

# Wrap the environment with your custom reward wrapper
env = MyRewardWrapper(env)

# === Match Simulation Parameters ===
num_episodes = 10
total_rewards = []
step_counts = []

logging.info(f"Starting {num_episodes} random agent matches...")

for episode in range(1, num_episodes + 1):
    obs = env.reset()
    done = False
    total_reward = 0
    step_count = 0

    logging.info(f"\n▶ Match {episode} started...")

    while not done:
        action = env.action_space.sample()
        obs, reward, done, info = env.step([action])
        total_reward = total_reward + reward
        step_count += 1
        time.sleep(0.01)  # Optional slowdown for observation

        #if step_count % 100 == 0:
        logging.info(f"Match {episode} | Step: {step_count} | Reward: {total_reward:.2f}")

    total_rewards.append(total_reward)
    step_counts.append(step_count)

    logging.info(f"✅ Match {episode} ended! Steps: {step_count} | Total reward: {total_reward:.2f}")

# === Cleanup ===
env.close()

# === Summary ===
logging.info("\n=== Summary of All Matches ===")
for i, (steps, reward) in enumerate(zip(step_counts, total_rewards), 1):
    logging.info(f"Match {i}: Steps = {steps}, Reward = {reward:.2f}")

logging.info(f"\nAverage reward: {sum(total_rewards)/num_episodes:.2f}")
logging.info(f"Average steps: {sum(step_counts)/num_episodes:.0f}")
logging.info(f"Logs saved to: {log_filename}")


### Now we get the reward using backprogation in a deep learning algo like PPO for example : 


In [ ]:
import gym
import logging
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from gfootball.env import create_environment

# === Logging setup ===
# === Setup Logging ===
log_filename = f"ppo_agent_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
logging.basicConfig(
    filename=log_filename,
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt='%H:%M:%S'
)
# === Create custom env ===
def make_env():
    env = create_environment(
        env_name="11_vs_11_stochastic",
        representation="simple115v2",
        render=False,
        logdir="dumps",
        write_full_episode_dumps=False,  # Do not generate videos during training
        write_video=False
    )
    env = MyRewardWrapper(env)
    return env

# Stable Baselines3 requires vectorized environments
env = DummyVecEnv([make_env])

# === Define PPO Model ===
model = PPO("MlpPolicy", env, verbose=1)


# === Train the agent ===
logging.info("\U0001F3C3 Starting training...")
model.learn(total_timesteps=100_000)

# === Save model ===
model.save("ppo_gfootball_agent")
logging.info("\U0001F4BE Model saved as 'ppo_gfootball_agent.zip'")

# === Create evaluation environment with video enabled ===
eval_env = create_environment(
    env_name="11_vs_11_stochastic",
    representation="simple115v2",
    render=False,
    logdir="dumps_eval",
    write_full_episode_dumps=True,
    write_video=True
)
eval_env = MyRewardWrapper(eval_env)

# === Evaluate the agent ===
obs = eval_env.reset()
done = False

episode_reward = 0
while not done:
    action, _ = model.predict(obs)
    obs, reward, done, info = eval_env.step(action)
    episode_reward += reward

logging.info(f"\U0001F3C6 Evaluation episode completed. Total reward: {episode_reward:.2f}")
eval_env.close()


# NEW REWARD SYSTEM :

In [ ]:
from enum import Enum

# Enum pour représenter les équipes
class BallOwnership(Enum):
    NOONE = -1
    LEFT = 0
    RIGHT = 1

# Enum pour les modes de jeu
class GameMode(Enum):
    NORMAL = 0
    KICKOFF = 1
    GOAL_KICK = 2
    FREE_KICK = 3
    CORNER = 4
    THROW_IN = 5
    PENALTY = 6

# Enum pour les indices précis de l'observation simple115v2
class Simple115v2Indices(Enum):
    LEFT_TEAM_POS = slice(0, 22)
    LEFT_TEAM_DIR = slice(22, 44)
    RIGHT_TEAM_POS = slice(44, 66)
    RIGHT_TEAM_DIR = slice(66, 88)
    BALL_POSITION = slice(88, 91)
    BALL_DIRECTION = slice(91, 94)
    BALL_OWNERSHIP = slice(94, 97)
    ACTIVE_PLAYER = slice(97, 108)
    GAME_MODE = slice(108, 115)


In [21]:
# Improve reward system : 

class MyRewardWrapper(gym.Wrapper):
    """
    Reward shaping wrapper for GFootball (simple115v2).

    Components:
      - Global MAX_REWARD with per-zone weighting
      - Ball-chasing (attack vs defend)
      - Anti-idleness penalty
      - Ball forward progression incentive
      - Goal detection bonus (not zone-scaled)
      - Micro-events (pass, forward-pass, dribble, shot) — zone-scaled
    """

    def __init__(
        self,
        env,
        *,
        max_reward=100.0,
        zone_weights=None,
        # chase / motion
        chase_attack_gain=2.0,
        chase_defend_gain=5.0,
        forward_gain=10.0,
        backward_penalty_gain=5.0,
        idle_move_threshold=0.001,
        idle_penalty=0.5,
        # micro-events (fractions of MAX_REWARD, then * zone_weight)
        w_pass=0.01,           # safe generic pass
        w_fwd_pass=0.02,       # forward pass (ball_x increases)
        w_dribble=0.03,        # sticky dribble
        w_shot_centre=0.10,    # shot in central attack zone
        w_shot_else=0.05,      # shot elsewhere
        # misc
        clamp_abs=100.0,
        min_forward_dx=0.05    # threshold to count as forward pass
    ):
        super().__init__(env)

        self.MAX_REWARD = float(max_reward)
        self.zone_weights = zone_weights or {
            "defense": 0.20,
            "milieu_def": 0.40,
            "milieu_att": 0.60,
            "attaque_ailes_g": 0.80,
            "attaque_ailes_d": 0.80,
            "attaque_centre": 1.00,
            None: 0.50,  # fallback
        }

        # motion/chase
        self.chase_attack_gain = float(chase_attack_gain)
        self.chase_defend_gain = float(chase_defend_gain)
        self.forward_gain = float(forward_gain)
        self.backward_penalty_gain = float(backward_penalty_gain)
        self.idle_move_threshold = float(idle_move_threshold)
        self.idle_penalty = float(idle_penalty)

        # micro events weights (fractions of MAX_REWARD, applied * zone_weight)
        self.w_pass = float(w_pass)
        self.w_fwd_pass = float(w_fwd_pass)
        self.w_dribble = float(w_dribble)
        self.w_shot_centre = float(w_shot_centre)
        self.w_shot_else = float(w_shot_else)

        self.clamp_abs = float(clamp_abs)
        self.min_forward_dx = float(min_forward_dx)

        self.prev_obs = None
        self.prev_score = [0, 0]

    # ---------- Gym API ----------

    def reset(self, **kwargs):
        self.prev_obs = self.env.reset(**kwargs)
        self.prev_score = [0, 0]
        return self.prev_obs

    def step(self, action):
        obs, base_reward, done, info = self.env.step(action)

        # Normalize action index (support int or [int])
        if isinstance(action, (list, tuple, np.ndarray)):
            action_idx = int(action[0])
        else:
            action_idx = int(action)

        raw_obs = info.get("raw_obs", {})

        # Context
        zone = self._get_zone(obs)
        zone_w = self.zone_weights.get(zone, self.zone_weights[None])
        ownership = np.argmax(obs[Simple115v2Indices.BALL_OWNERSHIP.value]) - 1  # -1:none, 0:us, 1:opp

        # Dense components (zone-scaled)
        chase_r   = self._chase_ball(obs, ownership)
        forward_r = self._forward_progress(obs)
        idle_r    = self._idle_penalty(obs)
        micro_r   = self._micro_events_reward(obs, action_idx, raw_obs, zone)

        shaped = zone_w * (chase_r + forward_r - idle_r + micro_r)

        # Sparse goal bonus (not zone-scaled)
        goals_r = self._goal_scored(info)

        total = base_reward + shaped + goals_r
        total = float(np.clip(total, -self.clamp_abs, self.clamp_abs))

        if logging.getLogger().isEnabledFor(logging.DEBUG):
            logging.debug(
                f"zone={zone} wz={zone_w:.2f} own={ownership} "
                f"base={base_reward:.2f} chase={chase_r:.3f} fwd={forward_r:.3f} "
                f"idle={idle_r:.3f} micro={micro_r:.3f} goals={goals_r:.1f} -> total={total:.2f}"
            )

        self.prev_obs = obs
        return obs, total, done, info

    # ---------- Reward components ----------

    def _chase_ball(self, obs, ownership):
        """Positive reward when active player reduces distance to ball.
        Stronger when defending (opponent owns the ball)."""
        if self.prev_obs is None:
            return 0.0

        ball_x, ball_y = obs[Simple115v2Indices.BALL_POSITION.value][:2]
        pball_x, pball_y = self.prev_obs[Simple115v2Indices.BALL_POSITION.value][:2]

        ppos = obs[Simple115v2Indices.LEFT_TEAM_POS.value].reshape(-1, 2)
        ppos_prev = self.prev_obs[Simple115v2Indices.LEFT_TEAM_POS.value].reshape(-1, 2)
        active_idx = int(np.argmax(obs[Simple115v2Indices.ACTIVE_PLAYER.value]))

        ax, ay = ppos[active_idx]
        pax, pay = ppos_prev[active_idx]

        prev_dist = np.linalg.norm([pball_x - pax, pball_y - pay])
        curr_dist = np.linalg.norm([ball_x - ax, ball_y - ay])
        dist_delta = prev_dist - curr_dist  # >0 if moved closer

        if ownership == 0:      # we own ball → mild incentive to stay close/support
            return max(0.0, dist_delta * self.chase_attack_gain)
        elif ownership == 1:    # opponent owns ball → stronger chase incentive
            return max(0.0, dist_delta * self.chase_defend_gain)
        else:
            return max(0.0, dist_delta * (0.5 * self.chase_attack_gain))

    def _forward_progress(self, obs):
        """Reward moving the BALL forward along +X (opponent goal at x≈+1).
        Small penalty for backward movement."""
        if self.prev_obs is None:
            return 0.0
        prev_x = float(self.prev_obs[Simple115v2Indices.BALL_POSITION.value][0])
        curr_x = float(obs[Simple115v2Indices.BALL_POSITION.value][0])
        dx = curr_x - prev_x
        if dx > 0:
            return dx * self.forward_gain
        elif dx < 0:
            return abs(dx) * (-self.backward_penalty_gain)
        return 0.0

    def _idle_penalty(self, obs):
        """Penalize if active player barely moves (to avoid frozen behavior)."""
        if self.prev_obs is None:
            return 0.0
        ppos = obs[Simple115v2Indices.LEFT_TEAM_POS.value].reshape(-1, 2)
        ppos_prev = self.prev_obs[Simple115v2Indices.LEFT_TEAM_POS.value].reshape(-1, 2)
        active_idx = int(np.argmax(obs[Simple115v2Indices.ACTIVE_PLAYER.value]))

        move = np.linalg.norm(ppos[active_idx] - ppos_prev[active_idx])
        return self.idle_penalty if move < self.idle_move_threshold else 0.0

    def _goal_scored(self, info):
        """Detect our team scoring using 'score' in info (list [us, opp])."""
        score = info.get("score", self.prev_score)
        reward = 0.0
        if score[0] > self.prev_score[0]:
            reward = 0.5 * self.MAX_REWARD  # e.g., +50 if MAX_REWARD=100
            logging.info(f"⚽ Goal! +{reward:.1f}")
        self.prev_score = score
        return reward

    # ---------- Micro-events (zone-scaled) ----------

    def _micro_events_reward(self, obs, action_idx, raw_obs, zone):
        """Small, dense rewards that teach football instincts."""
        if self.prev_obs is None:
            return 0.0

        r = 0.0

        # Successful pass (possession stayed in our team but ACTIVE player changed)
        if self._successful_pass(self.prev_obs, obs):
            r += self.w_pass * self.MAX_REWARD

        # Forward pass (ball advanced enough along X)
        if self._short_forward_pass(self.prev_obs, obs, self.min_forward_dx):
            r += self.w_fwd_pass * self.MAX_REWARD

        # Dribble (sticky action flag, index 9 in GRF)
        if self._dribble(raw_obs):
            r += self.w_dribble * self.MAX_REWARD

        # Shot taken (action index 12 in default action set)
        if self._shot_taken(action_idx):
            if zone == "attaque_centre":
                r += self.w_shot_centre * self.MAX_REWARD
            else:
                r += self.w_shot_else * self.MAX_REWARD

        return r

    # ---------- Micro helpers ----------

    def _successful_pass(self, prev_obs, obs):
        prev_team = np.argmax(prev_obs[Simple115v2Indices.BALL_OWNERSHIP.value]) - 1
        curr_team = np.argmax(obs[Simple115v2Indices.BALL_OWNERSHIP.value]) - 1
        if prev_team != 0 or curr_team != 0:
            return False
        prev_owner = int(np.argmax(prev_obs[Simple115v2Indices.ACTIVE_PLAYER.value]))
        curr_owner = int(np.argmax(obs[Simple115v2Indices.ACTIVE_PLAYER.value]))
        return prev_owner != curr_owner

    def _short_forward_pass(self, prev_obs, obs, min_dx=0.05):
        prev_x = float(prev_obs[Simple115v2Indices.BALL_POSITION.value][0])
        curr_x = float(obs[Simple115v2Indices.BALL_POSITION.value][0])
        return (curr_x - prev_x) > float(min_dx)

    def _dribble(self, raw_obs):
        # GRF sticky actions: index 9 is DRIBBLE (True = 1)
        sticky = raw_obs.get("sticky_actions") or [0]*10
        try:
            return int(sticky[9]) == 1
        except Exception:
            return False

    def _shot_taken(self, action_idx):
        # Default action set: 12 is SHOT
        return int(action_idx) == 12

    # ---------- Utilities ----------

    def _get_zone(self, obs):
        x, y = obs[Simple115v2Indices.BALL_POSITION.value][:2]
        if -1.0 <= x < -0.5:
            return "defense"
        elif -0.5 <= x < 0:
            return "milieu_def"
        elif 0 <= x < 0.5:
            return "milieu_att"
        elif 0.5 <= x <= 1.0:
            if -0.2 <= y <= 0.2:
                return "attaque_centre"
            elif y > 0.2:
                return "attaque_ailes_g"
            elif y < -0.2:
                return "attaque_ailes_d"
        return None


In [22]:
import os
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from gfootball.env import create_environment
from stable_baselines3.common.vec_env import DummyVecEnv


In [26]:
import gym
import stable_baselines3
import logging
print("Gym version:", gym.__version__)
print("Stable-Baselines3 version:", stable_baselines3.__version__)


Gym version: 0.21.0
Stable-Baselines3 version: 1.7.0


In [ ]:
# ===================== ACTION ADAPTER =====================
class SingleAgentActionAdapter(gym.Wrapper):
    """Ensures GFootball env accepts int actions from SB3 (wraps them as [action])"""
    def step(self, action):
        if isinstance(action, (list, tuple, np.ndarray)):
            return self.env.step(action)
        else:
            return self.env.step([int(action)])


# ===================== ENV CREATION =====================
def make_env(rank=0, seed=0):
    """Factory function for parallel environments"""
    def _init():
        env = create_environment(
            env_name="5_vs_5",         
            representation="simple115v2",
            render=False,
            logdir="dumps",
            write_full_episode_dumps=False,
            write_video=False
        )
        env = SingleAgentActionAdapter(env)
        env = MyRewardWrapper(env)
        env = Monitor(env)
        env.seed(seed + rank)
        return env
    return _init


# ===================== VECTORIZE ENV =====================
NUM_ENVS = 8  # 4 to 8 depending on your CPU
env_fns = [make_env(rank=i) for i in range(NUM_ENVS)]
vec_env = DummyVecEnv(env_fns)

# Normalize rewards for stability
vec_env = VecNormalize(vec_env, training=True, norm_obs=False, norm_reward=True)

# ===================== PPO MODEL =====================
logdir = "./ppo_football_logs"
os.makedirs(logdir, exist_ok=True)

model = PPO(
    "MlpPolicy",               # Multi-layer perceptron (fully connected NN)
    env=vec_env,
    verbose=1,
    tensorboard_log=logdir,
    n_steps=2048,              # rollout length per env
    batch_size=8192,           # combine experience from all envs
    n_epochs=10,               # mini-batch epochs per update
    gamma=0.995,               # long-term horizon (discount)
    gae_lambda=0.95,           # smoother advantage estimation
    clip_range=0.2,            # PPO clipping threshold
    ent_coef=0.005,            # exploration term
    learning_rate=3e-4,        # Adam optimizer LR
    vf_coef=0.5,               # value function loss weight
)

# ===================== CALLBACKS =====================
eval_env = SubprocVecEnv([make_env(seed=100+i) for i in range(2)])
eval_env = VecNormalize(eval_env, training=False, norm_obs=False, norm_reward=False)

eval_cb = EvalCallback(
    eval_env,
    best_model_save_path=os.path.join(logdir, "best"),
    log_path=os.path.join(logdir, "eval"),
    eval_freq=100_000 // NUM_ENVS,
    deterministic=True,
)

ckpt_cb = CheckpointCallback(
    save_freq=200_000 // NUM_ENVS,
    save_path=os.path.join(logdir, "checkpoints"),
    name_prefix="ppo_gfootball"
)

# ===================== TRAINING LOOP =====================
model.learn(total_timesteps=5_000_000, callback=[eval_cb, ckpt_cb])
model.save(os.path.join(logdir, "final_model"))
vec_env.save(os.path.join(logdir, "vec_normalize.pkl"))


Using cpu device
Logging to ./ppo_football_logs\PPO_2
------------------------------
| time/              |       |
|    fps             | 126   |
|    iterations      | 1     |
|    time_elapsed    | 129   |
|    total_timesteps | 16384 |
------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 3e+03        |
|    ep_rew_mean          | 326          |
| time/                   |              |
|    fps                  | 104          |
|    iterations           | 2            |
|    time_elapsed         | 313          |
|    total_timesteps      | 32768        |
| train/                  |              |
|    approx_kl            | 0.0012667975 |
|    clip_fraction        | 0.000513     |
|    clip_range           | 0.2          |
|    entropy_loss         | -2.94        |
|    explained_variance   | 0.0719       |
|    learning_rate        | 0.0003       |
|    loss                 | 0.0592       

KeyboardInterrupt: 